In [0]:
%run ../config_setup

# Silver 04 — Work Order

RDS CDC se `work_order` events clean karke latest state env-configured `work_order` silver table me overwrite karta hai (simple upsert, no SCD2).

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, get_json_object, expr

spark = SparkSession.builder.getOrCreate()

df = spark.readStream.table(f"{BRONZE}.rds_cdc").filter(col("table") == "work_order").select(
    col("_lsn"), col("_change_type"),
    get_json_object("after", "$.work_order_id").alias("work_order_id"),
    get_json_object("after", "$.machine_id").alias("machine_id"),
    get_json_object("after", "$.start_ts").alias("start_ts"),
    get_json_object("after", "$.end_ts").alias("end_ts"),
    get_json_object("after", "$.status").alias("status")
)

def upsert_wo(microBatchDF, batchId):
    if microBatchDF.count() == 0: return
    latest = microBatchDF.withColumn("rn", expr("row_number() over (partition by work_order_id order by _lsn desc)")).filter("rn = 1").drop("rn")
    latest.filter(col("_change_type") != "delete").drop("_lsn", "_change_type") \
          .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.work_order")

df.writeStream.foreachBatch(upsert_wo).option("checkpointLocation", f"{CHECKPOINTS}/silver_work_order_v2").trigger(availableNow=True).start().awaitTermination()

print("✅ silver_04_work_order done (start_ts & end_ts ke saath)!")